In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModel
import weaviate
import weaviate.classes as wvc
from weaviate.classes.query import Filter

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"  # NOT the article encoder — different training objective
COLLECTION_NAME = "AlzheimersChunk"
MAX_LENGTH = 64   # MedCPT queries are typically short (this is the model's expected query length — verify against model card)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
model = AutoModel.from_pretrained(QUERY_MODEL_NAME).to(DEVICE)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2528.37it/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [4]:
def embed_query(query_text: str):
    with torch.no_grad():
        encoded = tokenizer(
            [query_text],
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        outputs = model(**encoded)
        embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] pooling — same reasoning as article encoder
        return embedding.cpu().numpy()[0].tolist()

In [9]:
client = weaviate.connect_to_local()
collection = client.collections.get(COLLECTION_NAME)

In [6]:
def hybrid_search(query_text, top_k=10, alpha=0.5, section_type_filter=None, kwd_filter=None):
    """
    alpha: 0 = pure BM25 keyword search, 1 = pure vector search, 0.5 = balanced.
    Sweep this once you have real eval queries — don't trust 0.5 as final.
    """
    query_vector = embed_query(query_text)

    filters = None
    if section_type_filter:
        filters = Filter.by_property("section_type").equal(section_type_filter)
    if kwd_filter:
        kwd_f = Filter.by_property("kwd").contains_any([kwd_filter])
        filters = kwd_f if filters is None else (filters & kwd_f)

    response = collection.query.hybrid(
        query=query_text,          # feeds the BM25 half
        vector=query_vector,       # feeds the vector half — MedCPT query embedding
        alpha=alpha,
        limit=top_k,
        filters=filters,
        return_metadata=wvc.query.MetadataQuery(score=True, explain_score=True),
    )

    results = []
    for obj in response.objects:
        results.append({
            "chunk_id": obj.properties.get("chunk_id"),
            "text": obj.properties.get("text"),
            "doc_title": obj.properties.get("doc_title"),
            "heading_path": obj.properties.get("heading_path"),
            "section_type": obj.properties.get("section_type"),
            "score": obj.metadata.score,
            "explain_score": obj.metadata.explain_score,
        })
    return results

In [11]:
# query = "What is the role of amyloid-beta plaques in Alzheimer's progression?"
query = "What causes neurofibrillary tangles to form?"

hits = hybrid_search(query, top_k=5, alpha=0.5)

for i, hit in enumerate(hits, 1):
    print(f"\n[{i}] score={hit['score']:.4f} | {hit['doc_title']}")
    print(f"    section: {hit['section_type']} | path: {hit['heading_path']}")
    print(f"    {hit['text'][:200]}...")

client.close()


[1] score=0.8908 | 
    section:  | path: []
    (a) Formation of amyloid plaques: Aβ peptides, generated from the sequential cleavage of APP, initially exist as soluble monomers that misfold and aggregate into soluble oligomers and protofibrils, wh...

[2] score=0.8557 | 
    section:  | path: ['Hypothetical ischemic pattern of Alzheimer’s disease development']
    Alzheimer’s disease is characterized by progressive, massive neuronal loss, blood-brain barrier changes, neuroinflammation, deposition of extracellular amyloid plaques, intracellular neurofibrillary t...

[3] score=0.5880 | 
    section:  | path: ['Introduction']
    Alzheimer’s disease (AD), the most prevalent form of dementia, is clinically characterized by progressive memory decline. Post-mortem examination of AD brains consistently reveals hallmark pathologica...

[4] score=0.5495 | 
    section:  | path: ['FOCM and Alzheimer’s Disease']
    AD is a progressive neurodegenerative disorder that primarily affects individu

## Prompting

In [7]:
def assemble_child_context(hits):
    """
    Formats retrieved child chunks into a single context block for the LLM.
    Includes provenance (title, section path) so answers can be traced back —
    useful for debugging bad retrievals during eval later.
    """
    blocks = []
    for i, hit in enumerate(hits, 1):
        path = " > ".join(hit["heading_path"]) if hit["heading_path"] else hit["section_type"]
        block = (
            f"[Source {i}] {hit['doc_title']} | {path}\n"
            f"{hit['text']}"
        )
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)

def answer_with_child_context(query, top_k=5, alpha=0.5, section_type_filter=None):
    hits = hybrid_search(query, top_k=top_k, alpha=alpha, section_type_filter=section_type_filter)

    if not hits:
        return {"answer": None, "context": None, "hits": [], "note": "No chunks retrieved — check query or filters"}

    context = assemble_child_context(hits)

    system_prompt = (
        "You are a biomedical assistant answering questions about Alzheimer's disease research. "
        "Answer ONLY using the provided source excerpts below. "
        "If the excerpts don't contain enough information to answer, say so explicitly — "
        "do not use outside knowledge. "
        "Cite which [Source N] you drew each claim from."
    )

    user_message = f"Context:\n\n{context}\n\n---\n\nQuestion: {query}"

    combined_prompt = f"{system_prompt}\n\n{user_message}"
    return combined_prompt

In [10]:
# query = "What is the role of amyloid-beta plaques in Alzheimer's progression?"
query = "What causes neurofibrillary tangles to form?"
result = answer_with_child_context(query, top_k=5)

print(result)

You are a biomedical assistant answering questions about Alzheimer's disease research. Answer ONLY using the provided source excerpts below. If the excerpts don't contain enough information to answer, say so explicitly — do not use outside knowledge. Cite which [Source N] you drew each claim from.

Context:

[Source 1]  | 
(a) Formation of amyloid plaques: Aβ peptides, generated from the sequential cleavage of APP, initially exist as soluble monomers that misfold and aggregate into soluble oligomers and protofibrils, which are considered highly neurotoxic. These species further assemble into insoluble fibrils that accumulate extracellularly, forming dense-core amyloid plaques. Thus, Aβ aggregation involves a dynamic transition from soluble forms (monomers, oligomers, protofibrils) to insoluble fibrillar deposits, a key pathological hallmark of AD. (b) Formation of neurofibrillary tangles: the microtubule-associated protein tau undergoes abnormal hyperphosphorylation, leading to its det